# Decision tables and PRISM rules

Two model passes turn a predictable categorical target into small,
readable rule systems. Decision tables state that a small set of columns
determines the target, cross-validated. PRISM induces compact covering
rules — "if these conditions, then this class" — for the most predictable
targets. Their findings are `dependencies/predictive` and
`conditional_rules/if_then` statements respectively, and every rule is
re-measured by the shared evaluator before it can enter the report.

## Decision tables

For each of the most predictable categorical targets (top 6 on `standard`,
10 on `deep`), the pass searches for a small feature subset that predicts
the target through a lookup table: coarse feature codes (numeric bins
folded to 8 buckets, nominal levels kept), cell keys packed into a single
u64, cell-majority prediction with a global fallback. The search is
greedy over subsets of at most 3 features, drawn from the top 16 by model
gain, and each candidate subset is scored by 5-fold cross-validated
accuracy (the fold assignment is deterministic).

A table survives when CV accuracy ≥ 0.85 and at least 5 points above the
majority baseline. The finding names the subset and both accuracies — the
predictive twin of an approximate dependency, with the honest
out-of-sample error instead of a violation count.

## PRISM

PRISM (Cendrowska, 1987) induces modular rules one class at a time: start
from the rows of one class, repeatedly add the single condition that most
increases the class's share within the covered rows, emit the rule when
its precision crosses the bar, remove the covered rows, repeat. The
implementation evaluates candidate conditions with precomputed bitset
masks — a few hundred word-ANDs per candidate instead of row scans.

PRISM runs only on targets the predictability screen scored ≥ 0.9 with a
majority baseline under 0.9 (a near-majority target has nothing to
explain; rules for it would be lucky pockets, not structure) and at most
10 classes. Rules are capped at 3 per class, 4 conditions each, with
support ≥ 20 rows and precision ≥ 0.8, drawn from the top 12 features by
gain — never the target's own column. Numeric conditions quote the bin
edge, an actual data value, not a rounded display string.

## The shared rule evaluator

Whatever discovers a rule, the report only shows it after a common
re-measurement. The evaluator replays the rule's typed predicates on a
deterministic diagnostic partition — one fifth of the sampled rows,
chosen by a stable hash of the original row id — and requires:

- at least 30 eligible rows (rows where the whole rule is defined),
- at least 30 antecedent matches,
- at least 20 joint matches,
- a conditional rate above the outcome's baseline rate on the same
  partition, by at least 0.02 absolute **or** a lift of at least 1.5.

Surviving rules get the evaluator's evidence appended — eligible rows,
antecedent/outcome/joint counts, conditional rate, baseline rate, lift —
and their claim rewritten in those common units. The discovery method's
own numbers stay in its own evidence record: PRISM's score on uncovered
positives is not relabeled as validation precision. Rules that fail the
evaluator are dropped; the discovery counts are not a fallback.

In [ ]:
import json
import random

import polars as pl

import onceover

rng = random.Random(11)
n = 4_000
segment = ["premium" if rng.random() < 0.5 else "std" for _ in range(n)]
shipping = [
    "express" if (rng.random() < 0.95 if s == "premium" else rng.random() < 0.05)
    else "ground"
    for s in segment
]
df = pl.DataFrame({"segment": segment, "shipping": shipping})

report = onceover.profile(df)
rule = next(f for f in report.findings if f["category"] == "conditional_rules")
print("claim:", rule["claim"])
print()
for e in rule["evidence"]:
    m = e["metrics"]
    keys = [k for k in ("antecedent", "consequent", "conditions",
                        "train_confidence", "holdout_confidence",
                        "discovery_score", "newly_covered_positives",
                        "antecedent_count", "joint_count", "conditional_rate",
                        "baseline_rate", "lift")
            if k in m]
    parts = [f"{k}={m[k]:.3f}" if isinstance(m[k], float) else f"{k}={m[k]}" for k in keys]
    print(f"evidence from {e['method']['detector']:16s} basis={e['basis']:9s} "
          + " ".join(parts))

Reading the output: the same statement carries separate evidence from
each contributor, in common units where the evaluator supplies them and
in discovery units where only the miner measured them. The partition is
*diagnostic*: the miners may already have seen its rows during
discovery, and the evaluation participates in selection. It is not an
independent confirmation sample, and it carries no multiple-testing
guarantee — a hundred candidate rules were measured, and the survivors
are the interesting ones, not a corrected family.

Report-wide limits apply across all rule detectors: at most 20 rules, at
most 2 per outcome, configured through the selection policy.

## References

Cendrowska, J. (1987). PRISM: An algorithm for inducing modular rules.
*International Journal of Man-Machine Studies* 27(4). Kohavi, R. (1995).
The power of decision tables. *ECML*. The evaluator's counts-and-baseline
framing follows standard association-rule vocabulary (support,
confidence, lift).